# Running MESA grids with POSYDON

<div class='alert alert-info'>

In this tutorial, you will setup a POSYDON MESA working directory, build an HMS-HMS binary grid config file, adding the POSYDON defaults, and introduce a custom stopping condition to the MESA run. You will end up with a setup to run MESA grids with POSYDON.

</div>

The building blocks of POSYDON are the grids of single star and binary MESA simulations.
Since we need large grids for each phase, the ease of setting up MESA simulations is an important part of the POSYDON infrastructure. For most population synthesis science cases, the POSYDON default grids might be enough.
But as we start to link more and more population properties to intrinsic stellar and binary physics, additional grids will become more important in our understanding of binaries and their interactions.

So in this practicum we're specifically going to focus on running a "grid" of single stars and binaries, and how you can change their physics.
Because an actual grid run can take several days to complete based on the number of runs, we are going to submit only a few (or a couple) binaries and single stars to show you the process of submission.
The main steps in this practicum are:
1. Setting up a grid run with the POSYDON MESA inlists.
2. Adding your own MESA inlist changes on top of the POSYDON ones (this is no MESA summer school though, so nothing too complex)
3. Submitting a MESA run.

<div class='alert alert-warning'>
For a lot of the things discussed in this tutorial, you will have to use the terminal. 
<strong>The terminal commands are written in special markdown cells instead of code cells.</strong>
</div>

# Setting up the environment for MESA runs with POSYDON

Before setting up your own MESA runs, we need to initialize some additional environment variables.
MESA requires us to specify the location of the `MESASDK_ROOT`, `MESA_DIR`, and `OMP_NUM_THREADS`.
The cell below creates a file `load_MESA` in the current directory that you can `source` to automatically set these environment parameters.

Furthermore, POSYDON uses a custom version of r11701 MESA. This means two things:
1. The MESA inlist parameters are a bit older than the online documentation shows.
2. You will use the version of MESA we have already set up for you.

<details>
<summary><strong>Additional details on the file</strong></summary>

The file first loads `conda` and loads into the `posydon_env`.
Then it sets up the required environment variables for MESA and MESASDK:
- `MESA_DIR`, the MESA directory
- `OMP_NUM_THREADS`, the number of CPUs/threads MESA uses
- `MESASDK_ROOT`, the path to the installed MESA SDK

The `source $MESASDK_ROOT/bin/mesasdk_init.sh` loads the MESASDK configuration
into your terminal. Note that this script and setup is make for the Jupyterhub for 
the POSYDON school. If you want to use this later, more manual set up is required.

More information about running POSYDON MESA simulations can also be found in the [POSYDON documentation](https://posydon.org/POSYDON/latest/tutorials-examples/MESA-grids/running-grids.html).

</details>

In [3]:
%%writefile load_MESA
# Load conda + environment
source /opt/conda/etc/profile.d/conda.sh
conda activate posydon_env
###MESA
# set MESA_DIR to be the directory to which you downloaded MESA
export MESA_DIR=~/data/MESA/mesa-r11701_witheoschange_andwithreverseMTchange_fiximplicitmdot
# set OMP_NUM_THREADS to be the number of cores on your machine
export OMP_NUM_THREADS=4
# set up the MESA SDK
export MESASDK_ROOT=~/data/MESA/mesasdk
source $MESASDK_ROOT/bin/mesasdk_init.sh

Overwriting load_MESA


After running the previous cell, you should now have an extra file in the directory of the notebook.


<div class='alert alert-warning'>
<strong> Task 1: </strong>

Load (`source`) the file in a terminal. How has the terminal changed?

</div>


<details>
    <summary> <strong>Solution</strong> </summary>

```bash

source load_MESA

```

The terminal should now include `(posydon_env)` before the current folder, indicating
that the POSYDON environment has been loaded correctly.
</details>


<div class='alert alert-warning'>
<strong> Task 2: </strong>

Check if MESA and the MESASDK is correctly loaded by printing their paths in the terminal.

</div>


<details>
    <summary> <strong>Solution</strong> </summary>


```
echo $MESA_DIR

echo $MESASDK_ROOT
```
</details>


# POSYDON grid runs configuration file

POSYDON contains the command below to setup a grid of MESA runs.
In this section, we will cover the different components of this command and use it to set up a MESA run:

<center>

```bash
posydon-setup-grid
```

</center>

<div class='alert alert-warning'>
<strong> Task 3: </strong>

Run the command in the terminal. What do you see?

</div>

<details>
    <summary> <strong>Solution</strong> </summary>

```
usage: posydon-setup-grid [-h] --inifile INIFILE --grid-type
                          GRID_TYPE [--run-directory RUN_DIRECTORY]
                          [--submission-type SUBMISSION_TYPE]
                          [-n NPROC] [--verbose]
posydon-setup-grid: error: the following arguments are required: --inifile, --grid-type
```
</details>

Two arguments are required for setting a POSYDON MESA grid:
1. `--inifile`.
2. `--grid-type`. For this lab, we will use `--grid-type=fixed`, which allows us to give a `grid.csv` with initial (binary) parameter to run. For more information, [see the documentation](https://posydon.org/POSYDON/v2.2.5/components-overview/mesa-grids.html)

For this tutorial, we will also use the command `--submission-type=shell`.

<div class='alert alert-warning'>
<strong> Task 4: </strong>

Using the help function (`-h`) of the command, can you find what `--submission-type=shell` does? What other option is available?

</div>

<details>
    <summary> <strong>Solution</strong></summary>

We see the following when running the command with `-h`:
```
--submission-type SUBMISSION_TYPE
                        Options include creating a shell script or a slurm script
```
It can be used to create a shell script. The other option is `slurm`. 
`slurm` is used on HPC clusters and will not be used in this lab, but you can find more info in [the POSYDON documentation](https://posydon.org/POSYDON/latest/components-overview/mesa_grids/fixed.html).

</details>


With the `--submission-type` and `--grid-type` set, we can now create the configuration file (`.ini` file). POSYDON uses this `.ini` file to setup the MESA inlists, run_star_extras, and output for grid runs.

[POSYDON-code/POSYDON-MESA-INLIST](https://github.com/POSYDON-code/POSYDON-MESA-INLISTS) contains 
the run_*_extras.f and inlists used for the POSYDON Data Releases and several default examples.
Each branch contains different reruns and custom inlists. 

For this tutorial, we have set up a specific branch: `mb_practicum`.
The code below will copy the Github repository and copy an example config file
for an HMS-HMS grid (hydrogen + hydrogen main-sequence binary systems).

<div class='alert alert-warning'>
    <strong> Task 5: </strong>

Change the directory path to your own current working directory.

</div>

In [5]:
# Change this to your current working directory
CURRENT_DIR = "CHANGE/TO/WORKING/DIRECTORY"

In [12]:
# cloning the POSYDON-MESA-INLIST repository
POSYDON_INLISTS = f"{CURRENT_DIR}/POSYDON-MESA-INLISTS"
!git clone https://github.com/POSYDON-code/POSYDON-MESA-INLISTS.git --branch mb_practicum {POSYDON_INLISTS}

fatal: destination path 'tests/POSYDON-MESA-INLISTS' already exists and is not an empty directory.



<div class='alert alert-warning'>
    <strong> Task 6: </strong>

Let's make a new directory (`run`) and copy the `{POSYDON_INLISTS}/r11701/running_scripts/HMS-HMS_quest.ini` configuration file over.
</div>


<details>
    <summary> <strong>Solution</strong></summary>

```
!mkdir -p {CURRENT_DIR}/run
PATH_TO_YOUR_WORK_DIR=f"{CURRENT_DIR}/run"
!cp '{POSYDON_INLISTS}/r11701/running_scripts/HMS-HMS_quest.ini' '{PATH_TO_YOUR_WORK_DIR}/HMS-HMS_quest.ini'
```


</details>


<div class='alert alert-warning'>
    <strong> Task 7: </strong>

Let's open up the copied `HMS-HMS_quest.ini` file and look at the structure in the file. What are the four sections in the file?
</div>



<details>
    <summary> <strong>Solution</strong></summary>

The configuration file contains the following sections:

1. `[slurm]`
As the name suggests, this contains all parameters related to the HPC submission manager, slurm.
This includes the number of nodes, to run as a job-array, the partition, user, walltime, and email-notifications. In this school, we are using `shell` as our submission type, **as such you can ignore this section.**

2. `[mesa_inlists]`
This is where the real setup starts. In this section we determine how POSYDON builds the MESA inlists and what information is stored. We will spend most of this practicum here. 

3. `[mesa_extras]`
This section defines the MESA make files and what `run_star_extras.f` and `binary_star_extras.f` to use.

4. `[run_parameters]`
This section is used to input the location of the `grid.csv` file, which we will discuss later.

</details>


<div class='alert alert-warning'>
    <strong> Task 8: </strong>

Inside the  `HMS-HMS_quest.ini` file, look for `posydon_github_root`.
This line describes the local location of the `POSYDON-MESA-INLISTS` folder.

Replace the current line with the path to the `POSYDON-MESA-INLISTS` folder you
have created earlier.

</div>



<details>
    <summary> <strong>Solution</strong></summary>

This should be the same path as before as set in `POSYDON_INLISTS`.
The following should give you the right path, if you're stuck:
```python
print(POSYDON_INLISTS)
```

</details>

## Defining the initial conditions of a grid `[run_parameters]`

In the previous section, we've set up all the basic requirements for running a grid with POSYDON.
However, we have not yet defined what systems we want to run.
As such, if you try to run the following command, you will get an error!

<center>

```bash
posydon-setup-grid --submission-type=shell --grid-type=fixed --inifile=HMS_HMS_quest.ini
```

</center>

We have to create a file in which we define what initial properties as a binaries or single star, we want to run.
POSYDON knows about this file because of the `[run_parameters]` in the configuration file.

<div class='alert alert-warning'>
    <strong> Task 9: </strong>

Inside the  `HMS-HMS_quest.ini` file, look for the `[run_parameters]` section.
Can you find what parameter defines the name of the grid file?

</div>

<details>
    <summary> <strong>Solution</strong></summary>

The `grid` parameter and the name of the grid file has to be `grid_test.csv`.

</details>

The grid file needs to have the following format:

```
initial_z,Zbase,m1,m2,initial_period_in_days
0.00142,0.00142,30.,21.,10.
```

`initial_z` and `Zbase` are the metallicity in absolute metallicity of the model, which are the same here. We will use 1/10th solar here.
The other parameters should speak for themselves with `m1` and `m2` being in solar masses, and `initial_period_in_days` in days (as the name implies). These are specific parameters in the MESA inlists that will be replaced when running the grid.

<details>
<summary>Extra information</summary>

<strong>Comments on Metallicity</strong>
We define the metallicity in multiple places in the setup. The `initial_z` and `Zbase` in the grid are the first location we encounter them. The second location is in the `zams_filename` parameter as part of `[mesa_inlists]` later in the tutorial. **The metallicity and composition of the selected ZAMS model and the grid file need to be the same.** Otherwise, MESA will not be able to evolve the star! For this practicum, you should not have to change the `zams_filename`.

The POSYDON-MESA-INLISTS repository comes with ZAMS models for the 8 different metallicities in the POSYDON Data Releases.

</details>


The code below will create a 



<div class='alert alert-warning'>
    <strong> Task 10: </strong>

1. Replace the PATH_TO_YOUR_WORK_DIR below with your working directory with the `HMS_HMS_quest.ini` file. 
2. Give the grid file the correct name
</div>


In [ ]:
PATH_TO_YOUR_WORK_DIR = "PATH_TO_YOUR_WORK_DIR"

import csv
import os

grid_file = '????????'
path_to_file = os.path.join(PATH_TO_YOUR_WORK_DIR, grid_file)

with open(path_to_file, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['initial_z','Zbase','m1','m2','initial_period_in_days'])
    # Feel free to change the binary parameters: 
    # m1, m2, and initial_period_in_days here to test a different system
    # Please don't change the first two columns for this practicum.
    writer.writerow([0.00142, 0.00142, 30., 21., 10.])


## Building MESA with POSYDON

With the grid file in our run folder, we now have everything to do a MESA run.
Of course, we haven't changed any of the other parameters, but we will discuss
those in the next section.
<div class='alert alert-warning'>
    <strong> Task 11: </strong>

Let's go into our `run` directory and build the grid run setup with the following command:
</div>


```bash
posydon-setup-grid \
    --inifile=HMS-HMS_quest.ini \
    --grid-type=fixed \
    --submission-type=shell
```


A lot of input will be printed on your screen now.
First, the MESA files will be copied and then compiled.
Finally, some additional information about the grid parameters will be printed.

At the same time, a whole bunch of new files and folders will have populated your current working directory.
That's why it's advised to do each grid run in a separate folder!

<details>

<summary> <strong>Details on the produced files</strong> </summary>

A bunch of files and folders are created when you ran the previous command.
In short, this is what each of them does:

- binary: contains the MESA binary components
- columns_lists: contains the MESA columns configuration
- star1:  contains the MESA star1 components
- star2:  contains the MESA star2 components
- grid_command.sh: loop over the MESA runs 
- mk: makes MESA (already done)

</details>

Let's make sure we have just the basic MESA inlists and `run_star_extras.f`.

<div class='alert alert-warning'>
    <strong> Task 11: </strong>

Open `binary/src/run_star_extras.f` and check the number of lines.
The default MESA `run_star_extras.f` should have only ~41 lines of code.
</div>

<details>
    <summary> <strong>Solution</strong></summary>

```bash

cat binary/src/run_star_extras.f
```

</details>

Amazing!
In principle, you can now submit this grid run with a single system:
This will start the MESA run :) 

<center>

```bash
./grid_command.sh
```

</center>

You now have the most basic setup for running MESA grids with POSYDON.
If you have never run MESA before, this might be cool to do and see! However, the actual evolution of this system will take a while.
However, if you have run MESA before, you might want to change the parameters instead of running a default MESA model.

So in the next few sections, we will be covering more details on including your own inlists, `run_star_extras.f`, `run_binary_extras.f`, etc.
These are essential features we use in POSYDON to (re)run grids with a high convergence percentage.
We will also show how you can start from the POSYDON configuration and expand up on it.

# Custom MESA configuration

The POSYDON `HMS-HMS_quest.ini` configuration file contains two sections essential
for customizing your MESA runs:
1. `[mesa_extras]`
2. `[mesa_inlists]`

We will first look at `[mesa_extras]`, before looking into `[mesa_inlists]` later

## User defined `[mesa_extras]`

This section defines the `run_star_extras` and `run_binary_extras` used in the simulation. POSYDON uses a specific `run_star_extras` in the binary runs with additional termination flags for the post-processing.

`mesa_binary_extras` and `mesa_star_binary_extras` define what files are used in the binary runs.

`mesa_star1_extras` and `mesa_star2_extras` are used for single stars and pre-MS evolution, such as the CO-HeMS grid.
In the HMS-HMS grid run we are doing here, these are copied but not used in the evolution.

When you use the `user_binary_extras` and `user_star_binary_extras` parameters, these will overwrite
the `mesa_binary_extras` and `mesa_star_binary_extras` parameters.
Only one file will be used at the time and the `user` configuration will always be given priority!


With the information above, we're now going to load in the POSYDON MESA default binary and star extras!

<div class='alert alert-warning'>
    <strong> Task 12: </strong>

Let's uncomment the `user_binary_extras` and `user_star_binary_extras` in the `HMS-HMS_quest.ini` configuration file.
These new variables will overwrite the `mesa` defined variables.

Now rerun `posydon-setup-grid`. Afterwards, inspect `run_star_extras.f` and `run_binary_extras.f` in the generated folders.
What has changed?
</div>


<details>
    <summary> <strong>Solution</strong></summary>

```bash

cat run/binary/src/run_star_extras.f
cat run/binary/src/run_binary_extras.f
```

You should see the a `ReplaceValueWarning` showing up with `ReplaceValueWarning: 'Section mesa_extras value mesa_binary_extras is being set to None'`. This indicates that the `user` configuration is used instead.

</details>

You've now loaded in the POSYDON MESA defaults for the run_star and run_binary extras! 
You can change the path to any path you want and as such load in any `run_star_extras.f` or `run_binary_extras.f` that you want to use.

## User defined `[mesa_inlists]`

The `mesa_inlists` work a bit differently than the `mesa_extra`;
where only 1 `run_star_extras.f` and `run_binary_extras.f` can be used at the time,
multiple `inlists` can be stacked to create a combined final inlist.
Looking at the `HMS-HMS_quest.ini` file, these can be split into three levels of `inlist`'s:
1. The MESA r11701 default values: `binary_controls_mesa_defaults`
2. The POSYDON inlist: `binary_controls_posydon_defaults`
3. User defined inlists: `binary_controls_user`

Each exists for `job` & `controls` parameters and are stacked `1,2,3` with the 
latest being load in last and overwritting any previously defined variables.

In the configuration file, we are currently only defining the `mesa_defaults` values!


<details>
<summary> Additional details</summary> 

### `columns`

POSYDON requires special columns to be present in the history, binary and profile for the post-processing.
As such, we do not use the mesa defaults and instead directly point to the `POSYDON-MESA-INLISTS` to get the POSYDON default.
There is no stacking occuring for the columns definition.
Of course, you can alter the path and point to your own columns file(s).


### Additional parameters

There are a few more parameters in this section:

1. `scenario`, which is currently commented out. We will discuss at the end of the practicum.
2. `zams_filename`. As the name suggests, this file contains multiple ZAMS models at different masses for a specific metallicity configuration.
POSYDON contains 8 different metallicity ZAMS models. If you change the metallicity in the grid file, the ZAMS model also needs to be changed!
1. `single_star_grid`. Used to run a single star grid.

</details>

<div class='alert alert-warning'>
    <strong> Task 13: </strong>

We're going to rerun `posydon-setup-grid`, but now with the POSYDON defaults enabled.
Ignoring the `Star Formation` parameters, uncomment the other `posydon_defaults` parameters in the `HMS-HMS_quest.ini` file, for example `binary_controls_posydon_defaults`, `binary_jobs_posydon_defaults`, etc.. The POSYDON files contain both `_control` and `_job` components, thus they point to the same files. But you're not required to keep them in the same file.

Let's check out the new in `binary/inlist_project` file
</div>


<details>
    <summary> <strong>Solution</strong></summary>


We're changing too many parameters to list them, we show those in the beginning of 
`inlist_project` (in `&binary_controls`) and `inlist1` (`&controls`) files below.

<details>

<summary> `inlist_project` example differences </summary>

![Difference between MESA and POSYDON default inlist_project](./diff-mesa-posydon-inlist_project.png)

</details>

<details>

<summary> `inlist1` example differences </summary>

![Difference between MESA and POSYDON default inlist1](./diff-mesa-posydon-inlist1.png)

</details>

</details>

<div class='alert alert-info'>

If you see `initial_mass= 1` or `m1 = 1.0d0`, <strong>don't be alarmed</strong>. We replace the initial values for each binary when we submit the grid using `posydon-run-grid`. This happens behind the scenes, as long as `grid_test.csv` is correctly created. If you see `Grid parameters that effect binary_controls: m1,m2,initial_period_in_days` in the output of the `posydon-setup-grid`, everything is working correctly.
</div>

We have now reached one of the default POSYDON setup!
Although POSYDON now uses several reruns, this was one of the base inlists POSYDON started with.

### Custom user input

The last layer in the  POSYDON inlists is the `user` defined files, where you can define your own parameters on top of the MESA and POSYDON inlists.

<div class='alert alert-warning'>
    <strong> Task 14: </strong>


Running binary models can take a long time, therefore you are going to introduce an artificial stopping condition for the binary:
By creating a custom `user` MESA inlist and loading it into the grid building, can you stop the model at `model_number` 500?

</div>


<details>
    <summary> <strong>Solution</strong></summary>

`max_model_number` can be defined in the `star` controls, which stops the evolution after `X` models.

Write into a file:

```
&star_controls
    max_model_number = 500
/ ! end user controls
```

Then point to this file for your user star and binary controls inlists.

</details>



To incorporate you change, make sure to rerun the grid setup again!

```bash
posydon-setup-grid \
    --inifile=HMS-HMS_quest.ini \
    --grid-type=fixed \
    --submission-type=shell
```

<div class='alert alert-warning'>
    <strong> Task 15: </strong>


**Submit the grid with your custom end condition!**

</div>


Well done you've now submitted a first "grid" using POSYDON! Keep an eye on the run during the next part of the tutorial. You should be able to see if the simulation started and if reasonable output is/will be produced.




# Additional material on grid set up

There are a few parameters we haven't discussed in detail. We do not have the time to go into them for this practicum, but please ask one of the developers if you want to run a specific setup and want help with that.

## The `scenario` parameter

To allow for reproducible science, we have also added the `scenario` parameter.
It allows you to setup a specific branch and commit of the `POSYDON-MESA-INLISTS` github repository to use for the grid.
The `scenario` parameter consists of 3 different parameters:

1. `posydon` indicating what repository to use.
2. `mb_practicum-6968c2a2b0f0fad05da8cab1055d30ab3fd3d58a`, where `mb_practicum` is the branch name and `6968c2...` the commit you want to use. 
3. `HMS-HMS` defines what grid you want to run.

Although you can no longer define the `posydon` parameters when using the scenario. You are still able to overwrite paramters with the `user` inputs.

## Additional Parameters

We have not really touched upon a few parameters in the POSYDON grid config file, for example, `single_star_grid` or `star1/2_formation_job`.

- `single_star_grid` is used for running a single star grid and uses the single star termination conditions, which is especially useful when using a `scenario`. For example, using the `HMS-HMS` tag with `single_star_grid=True` will run H-rich single star models.

- The `star1/2_formation_job`s are for creating the ZAMS models and for creating the He stars in the CO-HeMS grids. 


## HPC facilities

The use of `--submission-type=shell` allows us to run MESA locally without requiring the use of a HPC facility. However, to produce large MESA grids, you will need more compute than available on your personal machine. Many HPC facilities use something called a scheduler, which puts jobs into a queue before they run. POSYDON currently supports `slurm` as an alternative `submission-type` with `--submission-type=slurm`.